# Comparación de métodos de verificación de FCMs

Este notebook ejecuta un experimento modular que compara cuatro métodos de verificación de espacios de estados para Mapas Cognitivos Difusos (FCM): **aritmético**, **simbólico**, **rectificado** y **lookahead**.

Incluye:
- Grid de mapas generados aleatoriamente con caché en `maps.json`
- Bucle de comparación con caché en `resultados.csv` y guardado de las series por configuración en `estados/*.npz`
- Métricas por método: covering, gap promedio, convergencia y validez
- Análisis de contención del intervalo final de lookahead frente a los demás métodos
- Análisis de encogimiento de las series (anidadas y estrictamente contrayentes)
- Soporte CPU/CUDA (`get_default_device`) y precisión configurable (`DTYPE`: `float64` o `float32`)

Ejecutar las celdas en orden, de arriba hacia abajo.

> Nota: para regenerar los resultados tras cambiar `DTYPE`, borra `resultados.csv` y `estados/`.

## Configuración

Parámetros del experimento y del grid de mapas. Ajustables libremente antes de ejecutar.

### Precisión numérica
- `DTYPE`: tipo de dato de todos los tensores. Cambiar `torch.float64` → `torch.float32` para más velocidad (especialmente en CUDA)

### Grid de generación de mapas
- `N_CONCEPTS_OPTIONS`: número de conceptos a probar
- `DENSITY_OPTIONS`: densidades de la matriz de influencia `W`
- `SEEDS`: seeds para reproducibilidad

### Parámetros del experimento
- `RESUME_TESTING`: si el CSV existe, reutilizarlo como caché
- `SAVE_EVERY_ITERS`: cada cuántas iteraciones se guarda
- `EPSILON`: tolerancia numérica
- `INPUT_LOWER_BOUND` / `INPUT_UPPER_BOUND`: cotas de los intervalos iniciales
- `ACTIVATION_FUNCTION`: función de activación (`sigmoid` o `tanh`)
- `NUM_SAMPLES`: muestras para la simulación (ground truth)
- `NUM_ITERS`: iteraciones de la serie
- `PHI_OPTIONS`: valores de `phi` a probar
- `LOOKAHEAD_K`: ventana de lookahead (`None` = cadena hasta el final, equivale a `k = NUM_ITERS`)
- `STATES_DIR`: directorio con un `.npz` por `(uuid, phi)` con las series de intervalos de cada iteración (un archivo por configuración)
- `MINIMAL_WARMING`: iteraciones de calentamiento (rectified)

In [ ]:
import torch

# <<< PRECISIÓN >>>
# torch.float64: precisión doble (máxima exactitud, lento en GPU de consumo)
# torch.float32: precisión simple (mucho más rápido en CUDA)
# Cambiar aquí afecta a todo el experimento (mapas, métodos y análisis).
DTYPE = torch.float64
torch.set_default_dtype(DTYPE)

MAPS_JSON = "maps.json"
CSV_FILE = "resultados.csv"
STATES_DIR = (
    "estados"  # intervalos por iteración y configuración (un .npz por (uuid, phi))
)

N_CONCEPTS_OPTIONS = [6, 12, 20]

# N_CONCEPTS_OPTIONS = [
#     2,
#     3,
#     4,
#     5,
#     6,
#     7,
#     8,
#     9,
#     10,
#     11,
#     12,
#     13,
#     14,
#     15,
#     16,
#     17,
#     18,
#     19,
#     20,
# ]
# DENSITY_OPTIONS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
DENSITY_OPTIONS = [0.3, 0.6, 0.9]
SEEDS = [42]
# SEEDS = [42, 123, 7]

RESUME_TESTING = True
SAVE_EVERY_ITERS = 20
EPSILON = 1e-6
INPUT_LOWER_BOUND = 0.0
INPUT_UPPER_BOUND = 1.0
ACTIVATION_FUNCTION = "sigmoid"
NUM_SAMPLES = 100
NUM_ITERS = 50
# PHI_OPTIONS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
PHI_OPTIONS = [0.3, 0.6, 1.0]
LOOKAHEAD_K = None  # None = cadena de lookahead hasta el final (k = NUM_ITERS)
MINIMAL_WARMING = 5

## Mapas (caché)

Genera o carga `maps.json` siguiendo la filosofía de caché:
- Si `MAPS_JSON` ya existe, se reutiliza.
- Si no, se generan los mapas (un mapa por `(n_concepts, density, seed)`, con seed fija por mapa) y se guardan.

In [ ]:
import json
import os

import torch

from core.utils import generate_random_matrix


def generate_maps(
    n_concepts_options: list[int],
    density_options: list[float],
    seeds: list[int],
) -> list[dict]:
    """Genera un mapa por (n_concepts, density, seed) con seed fija por mapa."""
    samples = []
    for n_concepts in n_concepts_options:
        for density in density_options:
            for seed in seeds:
                torch.manual_seed(seed)
                samples.append(
                    {
                        "uuid": f"n{n_concepts}_d{density:g}_s{seed}",
                        "n_concepts": n_concepts,
                        "density": density,
                        "weigths": generate_random_matrix(n_concepts, density).tolist(),
                    }
                )
    return samples


if os.path.exists(MAPS_JSON):
    print(f"Caché: {MAPS_JSON} ya existe, se reutiliza")
    with open(MAPS_JSON, "r") as f:
        samples = json.load(f)
else:
    print(f"{MAPS_JSON} no existe: generando mapas...")
    samples = generate_maps(N_CONCEPTS_OPTIONS, DENSITY_OPTIONS, SEEDS)
    with open(MAPS_JSON, "w") as f:
        json.dump(samples, f, indent=2)
    print(f"Guardados {len(samples)} mapas en {MAPS_JSON}")

print(f"Total de mapas cargados: {len(samples)}")

## Configuración de PyTorch

Selección de dispositivo, tipo de dato y modo de gradientes:
- `device`: `cuda` si está disponible, si no `cpu`
- `DTYPE`: definido en la celda de Configuración (`float64` por defecto; cambiar a `torch.float32` para GPU)
- Gradientes desactivados (`set_grad_enabled(False)`)
- Impresión de tensores con precisión extendida (`set_printoptions`)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)
torch.set_printoptions(precision=30, sci_mode=False)

print(f"Device: {device}, dtype: {DTYPE}")

## Imports — intervalo y métricas

Funciones básicas del núcleo: generación de entradas y muestras, redondeo hacia fuera, comprobaciones de contención, intersección de intervalos/series, cubrimiento, gap promedio y convergencia.

In [ ]:
import core
from core import (
    generate_inputs,
    generate_input_samples,
    upwards_rounding,
    downwards_rounding,
    outwards_rounding,
    is_inside_state,
    state_series_from_samples,
    is_inside_state_series,
    margin,
    states_intersection,
    series_intersection,
    covering,
    average_gap,
    convergence_iteration,
    series_is_nested,
    series_is_strictly_contracting,
)

## Imports — state-space simbólico

Funciones de relajación simbólica y construcción de series de estado: espacio de estados inducido, concretización, propagación, iteración simbólica/aritmética y los métodos `containment_guided`, `rectified` y `rescaled_reasoning` (referencia muestral).

In [ ]:
from core import (
    get_function_data,
    lower_relaxation,
    upper_relaxation,
    induced_state_space,
    concretize_state_space,
    propagate_state_space,
    apply_relaxed_activation,
    symbolic_iteration,
    arithmetic_iteration,
    containment_guided_state_space,
    rectified_intersection_state_space,
    interval_state_space,
    rescaled_reasoning,
    overapproximation_from_relaxations,
    symbolic_state_space,
    warm_start_symbolic_state_space,
)

## Registry de métodos

Para cambiar la comparación, edita `METHODS` (quitar/añadir métodos y activar o desactivar la comprobación de validez, clave `sound`) y `METHOD_ORDER`.

Cada método recibe `(ctx, states)` y devuelve la serie de estados `(max_iters + 1, 2, n_concepts)`:
- `ctx`: dict con `Wt`, `inputs`, `activation`, `phi`, `iters`, `device`, `eps`, `minimal_warming`, `lookahead_k`, ...
- `states`: dict con las series ya calculadas de los métodos anteriores en `METHOD_ORDER` (para resolver dependencias).

In [ ]:
from experimental.methods import lookahead_state_space


def run_arithmetic(ctx: dict, states: dict) -> torch.Tensor:
    return interval_state_space(
        ctx["Wt"],
        ctx["inputs"],
        ctx["activation"],
        ctx["phi"],
        ctx["iters"],
        device=ctx["device"],
    )


def run_symbolic(ctx: dict, states: dict) -> torch.Tensor:
    symbolic = symbolic_state_space(
        ctx["Wt"],
        ctx["inputs"],
        ctx["activation"],
        ctx["phi"],
        ctx["iters"],
        overapprox_threshold=None,
        device=ctx["device"],
    )
    return concretize_state_space(symbolic, ctx["inputs"], device=ctx["device"])


def run_rectified(ctx: dict, states: dict) -> torch.Tensor:
    baseline = series_intersection(states["arithmetic"], states["symbolic"])
    return rectified_intersection_state_space(
        ctx["Wt"],
        ctx["inputs"],
        ctx["activation"],
        ctx["phi"],
        ctx["iters"],
        minimal_warming_iters=ctx["minimal_warming"],
    )


def run_lookahead_10(ctx: dict, states: dict) -> torch.Tensor:
    return lookahead_state_space(
        ctx["Wt"],
        ctx["inputs"],
        ctx["activation"],
        ctx["phi"],
        k=10,
        max_iters=ctx["iters"],
        device=ctx["device"],
    )


def run_lookahead(ctx: dict, states: dict) -> torch.Tensor:
    return lookahead_state_space(
        ctx["Wt"],
        ctx["inputs"],
        ctx["activation"],
        ctx["phi"],
        k=ctx["lookahead_k"],
        max_iters=ctx["iters"],
        baseline_interval_series=states["arithmetic"],
        device=ctx["device"],
    )


METHODS = {
    "arithmetic": {"run": run_arithmetic, "sound": True},
    "symbolic": {"run": run_symbolic, "sound": True},
    "rectified": {"run": run_rectified, "sound": True},
    "lookahead_10": {"run": run_lookahead_10, "sound": True},
    "lookahead": {"run": run_lookahead, "sound": True},
}

METHOD_ORDER = ["arithmetic", "symbolic", "rectified", "lookahead_10", "lookahead"]

CONTAINMENT_MATRIX = False


def compute_methods(ctx: dict) -> dict:
    """Ejecuta todos los métodos en METHOD_ORDER resolviendo dependencias."""
    states: dict = {}
    for name in METHOD_ORDER:
        states[name] = METHODS[name]["run"](ctx, states)
    return states


def compute_stats(
    ctx: dict,
    states: dict,
    simulated_states: torch.Tensor,
    sampled_states: torch.Tensor,
) -> dict:
    """Calcula las métricas por método (covering, gap, convergencia, validez)."""
    stats = {
        "uuid": ctx["uuid"],
        "n_concepts": ctx["n_concepts"],
        "density": ctx["density"],
        "phi": ctx["phi"],
    }
    eps = ctx["eps"]

    for name in METHOD_ORDER:
        series = states[name]
        stats[f"{name}_covering"] = covering(series)
        stats[f"{name}_converged_at"] = convergence_iteration(series, eps)
        stats[f"{name}_average_gap"] = average_gap(series, simulated_states)
        if METHODS[name]["sound"]:
            stats[f"{name}_is_valid"] = is_inside_state_series(
                sampled_states, series, tolerance=eps
            )

    if CONTAINMENT_MATRIX:
        for a in METHOD_ORDER:
            for b in METHOD_ORDER:
                if a == b:
                    continue
                stats[f"{a}_inside_{b}"] = is_inside_state_series(
                    states[a], states[b], tolerance=eps
                )

    return stats

## Bucle de comparación

Recorre cada mapa y cada `phi`, ejecuta los métodos del registry y calcula las métricas.

1. Para cada `(uuid, phi)` se construye el `ctx` con los parámetros de configuración.
2. Se simula el ground truth muestral (`rescaled_reasoning`) y se calcula `sampled_covering`.
3. Se ejecutan `compute_methods(ctx)` y `compute_stats(...)`, acumulando los resultados.
4. El CSV se guarda cada `SAVE_EVERY_ITERS` muestras; Ctrl+C (`SIGINT`) detiene el bucle guardando lo procesado y `RESUME_TESTING` permite reanudar.

In [ ]:
from math import isnan
import os
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import signal

interrupted = False


def handler(signum, frame):
    global interrupted
    interrupted = True


signal.signal(signal.SIGINT, handler)


def _csv_row_is_complete(row: pd.Series) -> bool:
    """Fila de CSV válida: presentes y sin NaN las métricas esenciales."""
    essentials = (
        [f"{m}_covering" for m in METHOD_ORDER]
        + [f"{m}_average_gap" for m in METHOD_ORDER]
        + [f"{m}_is_valid" for m in METHOD_ORDER if METHODS[m]["sound"]]
        + ["sampled_covering"]
    )
    if not all(c in row.index for c in essentials):
        return False
    return bool(row[essentials].notna().all())


def _npz_is_valid(uuid: str, phi: float) -> bool:
    """El .npz de la configuración existe y contiene todos los métodos."""
    path = os.path.join(STATES_DIR, f"{uuid}_phi{phi:g}.npz")
    if not os.path.exists(path):
        return False
    try:
        with np.load(path) as data:
            return all(m in data for m in METHOD_ORDER)
    except Exception:
        return False


data = []
containment_results = []
dataframe = pd.DataFrame()
processed_sample_indexes = set()
csv_found = False

if RESUME_TESTING:
    try:
        dataframe = pd.read_csv(CSV_FILE, index_col=0)

        valid_rows = dataframe.apply(_csv_row_is_complete, axis=1)
        n_invalid = int((~valid_rows).sum())
        if n_invalid:
            print(
                f"CSV con {n_invalid} filas incompletas o inválidas (se recalcularán)"
            )

        # Solo se reutilizan configuraciones cuya fila CSV es válida Y cuyo .npz
        # de intervalos existe y contiene todos los métodos.
        processed_sample_indexes = {
            (uuid, phi)
            for uuid, phi in zip(
                dataframe.loc[valid_rows, "uuid"], dataframe.loc[valid_rows, "phi"]
            )
            if _npz_is_valid(uuid, phi)
        }
        csv_found = True
    except FileNotFoundError:
        print("No se encontró el archivo .csv")

# Corrida desde cero: limpiar estados guardados de corridas anteriores.
if not csv_found and os.path.isdir(STATES_DIR):
    shutil.rmtree(STATES_DIR)

os.makedirs(STATES_DIR, exist_ok=True)

try:
    for sample_index, sample in enumerate(tqdm(samples)):
        if interrupted:
            print("Detenido por el usuario.")
            break

        n_concepts = sample["n_concepts"]

        inputs = generate_inputs(
            n_concepts=n_concepts,
            l=INPUT_LOWER_BOUND,
            u=INPUT_UPPER_BOUND,
            device=device,
        )
        W = torch.tensor(sample["weigths"], device=device)
        Wt = W.T

        for phi in PHI_OPTIONS:
            if interrupted:
                break

            if (sample["uuid"], phi) in processed_sample_indexes:
                continue

            ctx = {
                "uuid": sample["uuid"],
                "n_concepts": n_concepts,
                "density": sample["density"],
                "phi": phi,
                "Wt": Wt,
                "inputs": inputs,
                "activation": ACTIVATION_FUNCTION,
                "iters": NUM_ITERS,
                "device": device,
                "eps": EPSILON,
                "minimal_warming": MINIMAL_WARMING,
                "lookahead_k": LOOKAHEAD_K,
            }

            # Aproximación muestral (ground truth)
            A = generate_input_samples(
                NUM_SAMPLES, n_concepts, INPUT_LOWER_BOUND, INPUT_UPPER_BOUND
            )
            simulated_states = rescaled_reasoning(
                A,
                W,
                T=NUM_ITERS,
                phi=phi,
                transfer_function=(
                    torch.sigmoid if ACTIVATION_FUNCTION == "sigmoid" else torch.tanh
                ),
                device=device,
            )
            sampled_states = state_series_from_samples(simulated_states)

            # Ejecutar métodos del registry y calcular métricas
            states = compute_methods(ctx)
            containment_results.append(
                {
                    "uuid": sample["uuid"],
                    "phi": phi,
                    "lookahead_inside_arithmetic_series": is_inside_state_series(
                        states["lookahead"], states["arithmetic"], tolerance=EPSILON
                    ),
                    "lookahead_inside_symbolic_series": is_inside_state_series(
                        states["lookahead"], states["symbolic"], tolerance=EPSILON
                    ),
                    "lookahead_inside_arithmetic_final": is_inside_state(
                        states["lookahead"][-1],
                        states["arithmetic"][-1],
                        tolerance=EPSILON,
                    ),
                    "lookahead_inside_symbolic_final": is_inside_state(
                        states["lookahead"][-1],
                        states["symbolic"][-1],
                        tolerance=EPSILON,
                    ),
                }
            )
            stats = compute_stats(ctx, states, simulated_states, sampled_states)
            stats["sampled_covering"] = covering(sampled_states)

            # Guardar series de intervalos por iteración (un .npz por configuración)
            path = os.path.join(STATES_DIR, f"{sample['uuid']}_phi{phi:g}.npz")
            np.savez_compressed(
                path,
                **{name: states[name].cpu().numpy() for name in METHOD_ORDER},
            )

            if any([isnan(x) for x in stats.values() if isinstance(x, float)]):
                print("NaN values found!!!")

            if any(
                not stats[f"{name}_is_valid"]
                for name in METHOD_ORDER
                if METHODS[name]["sound"]
            ):
                print("Validez: muestras fuera del intervalo de algún método!!!")

            data.append(stats)

        if (sample_index + 1) % SAVE_EVERY_ITERS == 0:
            dataframe = pd.concat([dataframe, pd.DataFrame(data)], ignore_index=True)
            dataframe.to_csv(CSV_FILE)
            data = []
finally:
    dataframe = pd.concat([dataframe, pd.DataFrame(data)], ignore_index=True)
    dataframe.to_csv(CSV_FILE)

## Análisis / diagnóstico de resultados

Reportes de diagnóstico sobre `dataframe` (resultados del bucle):

- **NaN**: filas con valores nulos en las métricas (`_covering` / `_average_gap`).
- **Validez**: métodos cuyas muestras simuladas (ground truth) quedan fuera del intervalo. `NaN` cuenta como fallo; se usa `== False` en vez de `~` porque las columnas pueden ser `object` (bool + NaN).
- **Contención**: pares `a` no contenido en `b` (solo si `CONTAINMENT_MATRIX` está activado).

Cada reporte se muestra únicamente si hay filas (`show_if_not_empty`).

In [ ]:
from IPython.display import display


def show_if_not_empty(dframe: pd.DataFrame) -> None:
    """Muestra el dataframe solo si tiene filas."""
    if not dframe.empty:
        display(dframe)


print("NaN Values Report Per Column:")
print(dataframe.isnull().sum())
print()

# Filas con NaN en métricas clave
metric_cols = [
    c
    for c in dataframe.columns
    if c.endswith("_covering") or c.endswith("_average_gap")
]
df_nans = dataframe[dataframe[metric_cols].isnull().any(axis=1)]
print(f"Filas con NaN en métricas: {len(df_nans)}")
show_if_not_empty(df_nans)

# Validez: métodos cuyas muestras simuladas (ground truth) quedan fuera.
# NaN cuenta como fallo (el método no produjo un intervalo verificable).
# Se usa == False en vez de ~ porque las columnas pueden ser object
# (bool + NaN) y ~ sobre Python bool/float da resultados erróneos.
valid_cols = [f"{name}_is_valid" for name in METHOD_ORDER if METHODS[name]["sound"]]
if valid_cols:
    not_sound = (dataframe[valid_cols].fillna(False) == False).any(axis=1)
    df_not_sound = dataframe[not_sound]
else:
    df_not_sound = dataframe.iloc[0:0]
print(
    f"Filas con fallos de validez (muestras fuera del intervalo): {len(df_not_sound)}"
)
show_if_not_empty(df_not_sound)

# Contención entre métodos (a no contenido en b). NaN cuenta como contención rota.
inside_cols = [c for c in dataframe.columns if "_inside_" in c]
if inside_cols:
    not_inside = (dataframe[inside_cols].fillna(False) == False).any(axis=1)
    df_not_inside = dataframe[not_inside]
else:
    df_not_inside = dataframe.iloc[0:0]
print(f"Filas con contención rota (a no contenido en b): {len(df_not_inside)}")
show_if_not_empty(df_not_inside)

## Gráficas

Una sola celda con toda la lógica de plotting. Métodos a graficar: los del registry más la envolvente muestral (`sampled`) si el CSV la incluye.

### Textos
- `METHODS_LABELS`: nombres de los métodos en las gráficas.
- `VAR_LABELS`: nombres de los ejes (`phi`, Número de neuronas, Conectividad).
- `METRIC_LABELS`: nombres de las métricas (Cubrimiento, Desviación).

### Configuración
- `GRID_PHI`, `GRID_N_CONCEPTS`, `GRID_DENSITY`: filtros de subplots (`None` = todos).
- `PLOT_COVERING_VS`, `PLOT_GAP_VS`, `PLOT_GAP_STD_PHI`, `PLOT_SURFACES`: qué graficar.
- `PLOT_Y_FROM_ZERO`: eje y (2D) desde 0 hasta `max(1, máximo)`.
- `PLOT_Z_FROM_ZERO`: eje de altura (3D) desde 0 hasta `max(1, máximo)`; colores del mapa normalizados a esa escala (0 → azul, ≥ 1 → rojo).

### Gráficas generadas
- Cubrimiento promedio (con min–max) variando `phi`, `n_concepts` y `density`.
- Desviación (gap) variando `phi`.
- Gap promedio ± desviación estándar por método.
- Superficies 3D por método (`gap` y `covering`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

sns.set(style="whitegrid")

PLOT_METHODS = list(METHOD_ORDER)
if "sampled_covering" in dataframe.columns:
    PLOT_METHODS += ["sampled"]

# ---------- Textos de las gráficas (labels) ----------
METHODS_LABELS = {
    "arithmetic": "Arithmetic",
    "symbolic": "Symbolic",
    "rectified": "Rectification",
    "lookahead": "Lookahead",
    "sampled": "Real Activations",
}
VAR_LABELS = {
    "phi": "phi",
    "n_concepts": "N. of Neurons",
    "density": "Density",
}
METRIC_LABELS = {
    "covering": "Covering",
    "average_gap": "Average Gap",
}
GAP_STD_PHI_LABEL = "Average Gap ± Standard deviation"

# ---------- Estilos de las líneas ----------
METHODS_COLORS = {
    "arithmetic": "#1f77b4",
    "symbolic": "#4B0082",
    "rectified": "#d62728",
    "lookahead": "#2ca02c",
    "sampled": "black",
}
METHODS_LINESTYLE = {"sampled": "--"}

# Filtros opcionales para los grids de subplots (None = todos)
GRID_PHI = None
GRID_N_CONCEPTS = None
GRID_DENSITY = None

# Qué graficar (True/False)
PLOT_COVERING_VS = ["phi", "n_concepts", "density"]
PLOT_GAP_VS = ["phi"]
PLOT_GAP_STD_PHI = True
PLOT_SURFACES = True

# Eje y (2D) desde 0 hasta max(1, valor máximo).
PLOT_Y_FROM_ZERO = True

# Eje de altura (3D) desde 0 hasta max(1, valor máximo) y colores del mapa
# normalizados a esa misma escala: 0 -> azul, 1 o más -> rojo.
PLOT_Z_FROM_ZERO = True

df = pd.read_csv(CSV_FILE)


def melt_metric(dframe: pd.DataFrame, metric: str) -> pd.DataFrame:
    """Formato largo: (uuid, n_concepts, density, phi, method, value)."""
    cols = {
        f"{m}_{metric}": m for m in PLOT_METHODS if f"{m}_{metric}" in dframe.columns
    }
    df_long = dframe.melt(
        id_vars=["uuid", "n_concepts", "density", "phi"],
        value_vars=list(cols),
        var_name="method",
        value_name="value",
    )
    df_long["method"] = df_long["method"].str.replace(f"_{metric}", "", regex=False)
    for col in ["phi", "value", "density", "n_concepts"]:
        df_long[col] = pd.to_numeric(df_long[col], errors="coerce")
    return df_long


def summarize(df_long: pd.DataFrame) -> pd.DataFrame:
    """Media/min/max de cada método por (n_concepts, density, phi)."""
    return (
        df_long.groupby(["n_concepts", "density", "phi", "method"], dropna=False)[
            "value"
        ]
        .agg(mean="mean", min="min", max="max")
        .reset_index()
    )


def filtered(df_stats: pd.DataFrame) -> pd.DataFrame:
    """Aplica los filtros GRID_* opcionales."""
    out = df_stats
    if GRID_PHI is not None:
        out = out[out["phi"].isin(GRID_PHI)]
    if GRID_N_CONCEPTS is not None:
        out = out[out["n_concepts"].isin(GRID_N_CONCEPTS)]
    if GRID_DENSITY is not None:
        out = out[out["density"].isin(GRID_DENSITY)]
    return out


def ylim_from_zero(ax: plt.Axes, values: pd.Series) -> None:
    """Eje y de 0 a max(1, máximo de values)."""
    if not PLOT_Y_FROM_ZERO:
        return
    ymax = max(1.0, float(values.max()))
    ax.set_ylim(0, ymax)


def plot_vs(
    df_stats: pd.DataFrame,
    x_col: str,
    row_col: str,
    col_col: str,
    metric: str,
    legend_loc: str = "best",
) -> None:
    """Subplots min-max por método: filas = row_col, columnas = col_col, eje x = x_col.

    Los textos (eje x, títulos, ylabel) salen de VAR_LABELS y METRIC_LABELS.
    """
    stats = filtered(df_stats)
    row_vals = np.sort(stats[row_col].unique())
    col_vals = np.sort(stats[col_col].unique())
    if len(row_vals) == 0 or len(col_vals) == 0:
        print("Sin datos para graficar.")
        return

    xlabel = VAR_LABELS.get(x_col, x_col)
    ylabel = METRIC_LABELS.get(metric, metric)

    fig, axes = plt.subplots(
        len(row_vals),
        len(col_vals),
        figsize=(6 * len(col_vals), 5 * len(row_vals)),
        squeeze=False,
    )
    for i, rv in enumerate(row_vals):
        for j, cv in enumerate(col_vals):
            ax = axes[i, j]
            sub = stats[(stats[row_col] == rv) & (stats[col_col] == cv)]
            if sub.empty:
                ax.axis("off")
                continue
            for m in PLOT_METHODS:
                df_m = sub[sub["method"] == m].sort_values(x_col)
                if df_m.empty:
                    continue
                ax.plot(
                    df_m[x_col],
                    df_m["mean"],
                    label=METHODS_LABELS.get(m, m),
                    color=METHODS_COLORS.get(m),
                    linestyle=METHODS_LINESTYLE.get(m, "-"),
                    linewidth=2,
                )
                ax.fill_between(
                    df_m[x_col],
                    df_m["min"],
                    df_m["max"],
                    alpha=0.25,
                    color=METHODS_COLORS.get(m),
                )
            ax.set_title(
                f"{VAR_LABELS.get(row_col, row_col)}={rv}  "
                f"{VAR_LABELS.get(col_col, col_col)}={cv}"
            )
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)
            ylim_from_zero(ax, sub[["mean", "min", "max"]].max(axis=1))
            ax.legend(loc=legend_loc, fontsize="small")
    plt.tight_layout()
    plt.show()


def surface3d(df_stats: pd.DataFrame, metric: str) -> None:
    """Superficie 3D por método: valor medio vs (n_concepts, density).

    Escala de colores y eje de altura: 0 -> azul, 1 o más -> rojo.
    """
    stats = filtered(df_stats)
    methods = [m for m in PLOT_METHODS if m in stats["method"].unique()]
    if len(methods) == 0:
        print("Sin datos para graficar 3D.")
        return

    zlabel = METRIC_LABELS.get(metric, metric)
    xlabel = VAR_LABELS.get("n_concepts", "n_concepts")
    ylabel = VAR_LABELS.get("density", "density")

    # Superficies de todos los métodos para calcular el máximo global
    surfaces = {}
    for m in methods:
        df_m = stats[stats["method"] == m]
        pivot = df_m.groupby(["n_concepts", "density"])["mean"].mean().reset_index()
        nc = np.sort(pivot["n_concepts"].unique())
        den = np.sort(pivot["density"].unique())
        X, Y = np.meshgrid(nc, den)
        Z = np.full_like(X, np.nan, dtype=float)
        for i, d in enumerate(den):
            for j, n in enumerate(nc):
                val = pivot[(pivot["n_concepts"] == n) & (pivot["density"] == d)][
                    "mean"
                ]
                if not val.empty:
                    Z[i, j] = val.values[0]
        surfaces[m] = (X, Y, Z)

    # Tope de la escala: 0 .. max(1, máximo global)
    z_axis_top = max(1.0, float(np.nanmax([surfaces[m][2] for m in methods])))

    fig = plt.figure(figsize=(6 * len(methods), 5))
    for idx, m in enumerate(methods, 1):
        X, Y, Z = surfaces[m]
        ax = fig.add_subplot(1, len(methods), idx, projection="3d")
        ax.plot_surface(
            X,
            Y,
            Z,
            cmap=cm.coolwarm,
            edgecolor="k",
            linewidth=0.2,
            alpha=0.9,
            vmin=0,
            vmax=z_axis_top,
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_zlabel(zlabel)
        if PLOT_Z_FROM_ZERO:
            ax.set_zlim(0, z_axis_top)
        ax.set_title(METHODS_LABELS.get(m, m))
    plt.tight_layout()
    plt.show()


# ==============================
# Generación de gráficas
# ==============================
covering_stats = summarize(melt_metric(df, "covering"))
gap_stats = summarize(melt_metric(df, "average_gap"))

# Cubrimiento promedio (con min-max) variando phi, n_concepts y density
for x_col in PLOT_COVERING_VS:
    if x_col == "phi":
        plot_vs(covering_stats, "phi", "n_concepts", "density", "covering")
    elif x_col == "n_concepts":
        plot_vs(covering_stats, "n_concepts", "phi", "density", "covering")
    else:
        plot_vs(covering_stats, "density", "n_concepts", "phi", "covering")

# Desviación (gap) variando phi
for x_col in PLOT_GAP_VS:
    if x_col == "phi":
        plot_vs(gap_stats, "phi", "n_concepts", "density", "average_gap")

# Promedio ± desviación estándar del gap por método, variando phi
if PLOT_GAP_STD_PHI:
    gap_long = melt_metric(df, "average_gap")
    gap_by_phi = (
        gap_long.groupby(["phi", "method"], dropna=False)["value"]
        .agg(mean="mean", std="std")
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    for m in [m for m in PLOT_METHODS if m != "sampled"]:
        df_m = gap_by_phi[gap_by_phi["method"] == m]
        if df_m.empty:
            continue
        ax.plot(
            df_m["phi"],
            df_m["mean"],
            label=METHODS_LABELS.get(m, m),
            color=METHODS_COLORS.get(m),
            linewidth=2,
        )
        ax.fill_between(
            df_m["phi"],
            df_m["mean"] - df_m["std"],
            df_m["mean"] + df_m["std"],
            alpha=0.25,
            color=METHODS_COLORS.get(m),
        )
    ax.set_xlabel(VAR_LABELS.get("phi", "phi"))
    ax.set_ylabel(GAP_STD_PHI_LABEL)
    ylim_from_zero(ax, gap_by_phi["mean"] + gap_by_phi["std"])
    ax.legend()
    plt.tight_layout()
    plt.show()

# Superficies 3D por método
if PLOT_SURFACES:
    surface3d(gap_stats, "average_gap")
    surface3d(covering_stats, "covering")

## Lookahead completo: rangos finales más ajustados

Comprueba la propiedad clave del método lookahead con la cadena hasta el final (`k = max_iters`): para cada configuración `(uuid, phi)`, compara el **intervalo final** (última iteración) de lookahead con el de cada uno de los demás métodos.

- **`contained`**: el rango final de lookahead está completamente contenido en el del método comparado (`is_inside_state`, con tolerancia `EPSILON`).
- **`tighter`**: contenido **y** estrictamente más estrecho (anchura total menor, es decir, realmente más ajustado).

Las series de intervalos por iteración las guarda el bucle de comparación en `STATES_DIR` (un `.npz` por configuración, con todos los métodos). Este análisis lee esos archivos y toma `[-1]` (el intervalo final) **sin recalcular** los métodos.


In [ ]:
# Análisis de contención leyendo los intervalos cacheados por el bucle
# (un .npz por configuración en STATES_DIR), sin recalcular los métodos.

import glob


def _load_state(data: np.lib.npyio.NpzFile, name: str) -> torch.Tensor:
    """Devuelve el intervalo final (última iteración) del método name."""
    series = torch.from_numpy(data[name]).to(device)
    return series[-1]


if not os.path.isdir(STATES_DIR):
    raise FileNotFoundError(
        f"{STATES_DIR} no existe: corre primero el bucle de comparación."
    )

paths = sorted(glob.glob(os.path.join(STATES_DIR, "*.npz")))
if not paths:
    raise FileNotFoundError(f"No hay archivos .npz en {STATES_DIR}.")

comparisons = []
skipped = 0
for path in tqdm(paths):
    base = os.path.basename(path)[: -len(".npz")]
    uuid, phi_str = base.rsplit("_phi", 1)
    phi = float(phi_str)

    with np.load(path) as data:
        if not all(m in data for m in METHOD_ORDER):
            print(f"Advertencia: {base} no contiene todos los métodos, se omite")
            skipped += 1
            continue
        last = {name: _load_state(data, name) for name in METHOD_ORDER}

    density = next(s["density"] for s in samples if s["uuid"] == uuid)
    n_concepts = last["arithmetic"].shape[1]
    width_lookahead = float((last["lookahead"][1] - last["lookahead"][0]).sum())

    for name in METHOD_ORDER:
        if name == "lookahead":
            continue
        other = last[name]
        contained = is_inside_state(last["lookahead"], other, tolerance=EPSILON)
        width_other = float((other[1] - other[0]).sum())
        tighter = contained and width_lookahead < width_other - EPSILON
        comparisons.append(
            {
                "uuid": uuid,
                "n_concepts": n_concepts,
                "density": density,
                "phi": phi,
                "method": name,
                "width_lookahead": width_lookahead,
                "width_other": width_other,
                "contained": bool(contained),
                "tighter": bool(tighter),
            }
        )

if skipped:
    print(f"Archivos omitidos por incompletos: {skipped}")

df_containment = pd.DataFrame(comparisons)
print(
    f"Configuraciones evaluadas: {df_containment['uuid'].nunique()} "
    f"({len(df_containment)} comparaciones lookahead vs otro método)"
)

In [ ]:
# Resumen: ¿el rango final de lookahead está contenido en el de cada método?

summary = df_containment.groupby("method").agg(
    n_total=("contained", "count"),
    n_contained=("contained", "sum"),
    n_tighter=("tighter", "sum"),
    mean_width_lookahead=("width_lookahead", "mean"),
    mean_width_other=("width_other", "mean"),
)
summary["frac_contained"] = summary["n_contained"] / summary["n_total"]
summary["frac_tighter"] = summary["n_tighter"] / summary["n_total"]
summary = summary.round(4)
display(summary)

not_contained = df_containment[~df_containment["contained"]]
if not_contained.empty:
    print(
        "Lookahead está contenido en TODOS los demás métodos en todas las configuraciones."
    )
else:
    print(f"Casos en que lookahead NO está contenido: {len(not_contained)}")
    display(not_contained[["uuid", "n_concepts", "density", "phi", "method"]])

# Desglose: nº de configuraciones en que lookahead está contenido, por método y n_concepts
breakdown = (
    df_containment.groupby(["method", "n_concepts"], dropna=False)
    .agg(n_contained=("contained", "sum"), n_tighter=("tighter", "sum"))
    .reset_index()
)
print("\nContenido (lookahead ⊆ método) por método y n_concepts:")
display(breakdown.pivot(index="method", columns="n_concepts", values="n_contained"))
print("\nEstrictamente más ajustado por método y n_concepts:")
display(breakdown.pivot(index="method", columns="n_concepts", values="n_tighter"))

## Encogimiento de las series

Una serie se considera que cumple el **Teorema del Encogimiento Débil** si el intervalo de cada iteración está completamente contenido en el de la iteración anterior:

```
states[i] ⊆ states[i-1]  para todo i = 1 .. max_iters
```

Es decir, los intervalos nunca crecen: se contraen (o se estabilizan) iteración a iteración. Una serie **encoge estrictamente** si, además, la anchura del intervalo disminuye en cada iteración (`series_is_nested` / `series_is_strictly_contracting`, con tolerancia `EPSILON`). Se lee de los `.npz` cacheados en `STATES_DIR` (guardados por el bucle), sin recalcular nada. Un intervalo con `NaN` cuenta como no anidado.

In [ ]:
# Convergencia de las series: para cada configuración y método,
# ¿el intervalo de cada iteración está contenido en el de la anterior?
# (anidada) y ¿la anchura disminuye en cada iteración? (estrictamente contrayente).


def _to_state_tensor(series: np.ndarray) -> torch.Tensor:
    """Convierte una serie guardada en .npz a tensor en device."""
    return torch.from_numpy(series).to(device)


if not os.path.isdir(STATES_DIR):
    raise FileNotFoundError(
        f"{STATES_DIR} no existe: corre primero el bucle de comparación."
    )

paths = sorted(glob.glob(os.path.join(STATES_DIR, "*.npz")))
if not paths:
    raise FileNotFoundError(f"No hay archivos .npz en {STATES_DIR}.")

convergence_rows = []
skipped = 0
for path in tqdm(paths):
    base = os.path.basename(path)[: -len(".npz")]
    uuid, phi_str = base.rsplit("_phi", 1)
    phi = float(phi_str)

    with np.load(path) as data:
        if not all(m in data for m in METHOD_ORDER):
            print(f"Advertencia: {base} no contiene todos los métodos, se omite")
            skipped += 1
            continue
        for name in METHOD_ORDER:
            series = _to_state_tensor(data[name])
            convergence_rows.append(
                {
                    "uuid": uuid,
                    "phi": phi,
                    "method": name,
                    "n_iters": series.shape[0],
                    "nested": series_is_nested(series, tolerance=EPSILON),
                    "strictly_contracting": series_is_strictly_contracting(
                        series, epsilon=EPSILON
                    ),
                }
            )

df_convergence = pd.DataFrame(convergence_rows)
if skipped:
    print(f"Archivos omitidos por incompletos: {skipped}")

# Cuántas series convergen por método (anidadas y estrictamente contrayentes)
summary = (
    df_convergence.groupby("method")
    .agg(
        n_total=("nested", "count"),
        n_nested=("nested", "sum"),
        n_strictly_contracting=("strictly_contracting", "sum"),
    )
    .reset_index()
)
summary["frac_nested"] = (summary["n_nested"] / summary["n_total"]).round(4)
summary["frac_strict"] = (summary["n_strictly_contracting"] / summary["n_total"]).round(
    4
)
display(summary)

# Desglose por método y n_concepts (series anidadas)
uuid_n = {s["uuid"]: s["n_concepts"] for s in samples}
df_convergence["n_concepts"] = df_convergence["uuid"].map(uuid_n)
print("\nSeries anidadas por método y n_concepts:")
display(
    df_convergence.pivot_table(
        index="method", columns="n_concepts", values="nested", aggfunc="sum"
    )
)

## Contención de lookahead respecto a los métodos base

Compara las series calculadas en memoria durante el bucle, sin leer los archivos `.npz`.

Se comprueba si lookahead está contenido en arithmetic y symbolic durante toda la serie y en el intervalo final.

In [ ]:
containment_df = pd.DataFrame(containment_results)
if containment_df.empty:
    print("No hay resultados en memoria. Ejecuta primero el bucle de comparación.")
else:
    print(f"Configuraciones comparadas en memoria: {len(containment_df)}")
    display(
        containment_df[
            [
                "lookahead_inside_arithmetic_series",
                "lookahead_inside_symbolic_series",
                "lookahead_inside_arithmetic_final",
                "lookahead_inside_symbolic_final",
            ]
        ]
        .sum()
        .to_frame("n_contained")
    )

In [ ]:
# --- Comparación lookahead (k=None, "Full") vs lookahead_10 (k=10) por n_concepts ---
# Para cada número de conceptos, sobre todos los casos con ese n_concepts:
#   - covering medio de cada método (menor = intervalo más ajustado/mejor)
#   - máxima diferencia absoluta |cov_Full - cov_k10| entre ambos métodos
#   - runtime medio de cada método (segundos)

import numpy as np
import pandas as pd
from IPython.display import display

if "CSV_FILE" not in globals():
    CSV_FILE = "resultados.csv"
df = pd.read_csv(CSV_FILE)

COV_FULL = "lookahead_covering"
COV_K10 = "lookahead_10_covering"
RT_FULL = "lookahead_runtime_s"
RT_K10 = "lookahead_10_runtime_s"

sub = df.dropna(subset=[COV_FULL, COV_K10, RT_FULL, RT_K10]).copy()
sub["diff_cov"] = (sub[COV_FULL] - sub[COV_K10]).abs()

rows = []
for nc, g in sub.groupby("n_concepts"):
    i_argmax = g["diff_cov"].idxmax()
    row = g.loc[i_argmax]
    rows.append(
        {
            "n_concepts": nc,
            "n_casos": len(g),
            "cov_Full_media": g[COV_FULL].mean(),
            "cov_k10_media": g[COV_K10].mean(),
            "max_|dcov|": g["diff_cov"].max(),
            "argmax (uuid,phi)": f"{row['uuid']} phi={row['phi']}",
            "runtime_Full_media_s": g[RT_FULL].mean(),
            "runtime_k10_media_s": g[RT_K10].mean(),
        }
    )

tabla = pd.DataFrame(rows).set_index("n_concepts")
tabla_str = tabla.to_string(float_format=lambda v: f"{v:.6f}")
print(tabla_str)

print(f"\nGlobal: covering medio  Full={sub[COV_FULL].mean():.6f} | k10={sub[COV_K10].mean():.6f}")
print(f"Global: runtime medio   Full={sub[RT_FULL].mean():.6f}s | k10={sub[RT_K10].mean():.6f}s")
print(f"Global: casos con covering idéntico (|d|<=1e-12): {(sub['diff_cov'] <= 1e-12).mean():.1%}")
